In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
%%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
DATA_ROOT = "/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}
import pandas as pd

In [4]:
### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # Parse dates on read to push conversion onto the GPU
    df = pd.read_csv(
        data_path,
        sep='|',
        parse_dates=['L_SHIPDATE', 'L_RECEIPTDATE', 'L_COMMITDATE'],
        **storage_options
    )
    print(df.columns)
    return df

lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [5]:
### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = f"{data_folder}/orders.tbl"
    # parse O_ORDERDATE on GPU during CSV read to avoid a separate datetime conversion
    df = pd.read_csv(
        data_path,
        sep="|",
        parse_dates=["O_ORDERDATE"],
        **storage_options
    )
    return df

orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [10]:
%%time
### cell 2 ###

# Filter and select L_ORDERKEY on GPU in one step
flineitem = lineitem.loc[
    lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE,
    ["L_ORDERKEY"]
]

# Filter orders by date range and select needed columns on GPU in one step
forders_opt = orders.loc[
    (orders.O_ORDERDATE >= "1993-08-01") & (orders.O_ORDERDATE < "1993-11-01"),
    ["O_ORDERKEY", "O_ORDERPRIORITY"]
]

# Perform the join and group/count entirely on GPU
total_opt = (
    forders
      .merge(flineitem, left_on="O_ORDERKEY", right_on="L_ORDERKEY")
      .groupby("O_ORDERPRIORITY", as_index=False)
      .O_ORDERKEY.count()
)

CPU times: user 149 ms, sys: 243 ms, total: 392 ms
Wall time: 340 ms


In [7]:
date1 = pd.Timestamp("1993-11-01")
date2 = pd.Timestamp("1993-08-01")
lsel = lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE
osel = (orders.O_ORDERDATE < date1) & (orders.O_ORDERDATE >= date2)
flineitem = lineitem[lsel]
forders = orders[osel]
jn = forders[forders["O_ORDERKEY"].isin(flineitem["L_ORDERKEY"])]
total = (
    jn.groupby("O_ORDERPRIORITY", as_index=False)["O_ORDERKEY"].count()
    # skip sort when Mars enables sort in groupby
    # .sort_values(["O_ORDERPRIORITY"])
)

In [12]:
forders

,O_ORDERKEY,O_CUSTKEY,O_ORDERSTATUS,O_TOTALPRICE,O_ORDERDATE,O_ORDERPRIORITY,O_CLERK,O_SHIPPRIORITY,O_COMMENT,O_DUMMY
2,3,1233140,F,270741.97,1993-10-14,5-LOW,Clerk#000009543,0,sly final accounts boost. carefully regular id...,<NA>
8,33,669580,F,126998.88,1993-10-27,3-MEDIUM,Clerk#000004086,0,uriously. furiously final request,<NA>
48,193,790601,F,78448.35,1993-08-08,1-URGENT,Clerk#000000246,0,the furiously final pin,<NA>
61,230,1025342,F,166252.82,1993-10-27,1-URGENT,Clerk#000005200,0,odolites. carefully quick requ,<NA>
63,256,1248304,F,154160.20,1993-10-19,4-NOT SPECIFIED,Clerk#000008334,0,he fluffily final ideas might are final accoun...,<NA>
...,...,...,...,...,...,...,...,...,...,...
14999859,59999428,1010578,F,153555.56,1993-08-27,3-MEDIUM,Clerk#000008590,0,ckages haggle furiously quickly regular,<NA>
14999894,59999559,140266,F,294263.33,1993-08-27,2-HIGH,Clerk#000001242,0,to the courts. slyly careful instructions sle...,<NA>
14999906,59999619,844771,F,106197.20,1993-09-12,5-LOW,Clerk#000002457,0,sts along the regular depths shall haggle blit...,<NA>
14999951,59999808,786487,F,296921.38,1993-09-06,4-NOT SPECIFIED,Clerk#000007773,0,eans haggle according to the slyly even foxes....,<NA>


In [9]:
total

,O_ORDERPRIORITY,O_ORDERKEY
0,1-URGENT,104689
1,2-HIGH,105021
2,3-MEDIUM,105198
3,4-NOT SPECIFIED,105487
4,5-LOW,105303
